In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
def create_state(mesh):
    rng_seed = 2

    # Create fine mesh for smooth nonlinear solution
    #mesh = Mesh(unit_square.GenerateMesh(maxh=0.025))

    # Finite element spaces
    fes_psi = H1(mesh, order=3, dirichlet=".*")
    fes_omega = H1(mesh, order=3, dirichlet=".*")

    # Trial and test functions
    psi, vpsi = fes_psi.TnT()
    omega, vomega = fes_omega.TnT()

    settings = {"Objects": {"Wireframe": False}, "Colormap" : { "ncolors" : 256 }}

    # Spatial coordinates
    #x, y = x, y

    rng = np.random.default_rng(rng_seed)

    # Random source term for vorticity
    f = GridFunction(fes_omega)
    f.vec.FV().NumPy()[:] = rng.normal(size=fes_omega.ndof)

    # Nonlinear diffusion parameter (higher = more nonlinear effects)
    nu = .1

    # Step 1: Solve for vorticity using Newton's method with nonlinear advection
    gf_omega = GridFunction(fes_omega, "omega")


    def SolveNonlinearVorticity(gf_omega, num_iterations=15):
        """
        Solve the nonlinear vorticity equation: -νΔω + (u·∇)ω = f
        using Picard iteration with linearization of convection term
        """
        for iteration in range(num_iterations):
            # Current streamfunction to get velocity
            gf_psi_temp = GridFunction(fes_psi)
            a_psi_temp = BilinearForm(fes_psi)
            a_psi_temp += grad(psi) * grad(vpsi) * dx
            a_psi_temp.Assemble()
            l_psi_temp = LinearForm(fes_psi)
            l_psi_temp += gf_omega * vpsi * dx
            l_psi_temp.Assemble()
            gf_psi_temp.vec.data = a_psi_temp.mat.Inverse(fes_psi.FreeDofs()) * l_psi_temp.vec

            # Velocity field: u = (∂ψ/∂y, -∂ψ/∂x)
            u_vel = grad(gf_psi_temp)[1]
            v_vel = -grad(gf_psi_temp)[0]

            # Bilinear form with nonlinear advection term
            a = BilinearForm(fes_omega)
            a += nu * grad(omega) * grad(vomega) * dx

            # Nonlinear convection (u·∇)ω linearized around previous iterate
            # (u·∇)ω is nonlinear but we linearize it using the velocity from previous step
            a += (u_vel * grad(omega)[0] + v_vel * grad(omega)[1]) * vomega * dx
            a.Assemble()

            # Linear form
            l = LinearForm(fes_omega)
            l += f * vomega * dx
            l.Assemble()

            # Residual and convergence check
            res = l.vec - a.mat * gf_omega.vec
            res_norm = Norm(res)

            if iteration == 0:
                print(f"Iteration {iteration}: residual norm = {res_norm:.6e}")
            elif iteration % 3 == 0:
                print(f"Iteration {iteration}: residual norm = {res_norm:.6e}")

            if res_norm < 1e-9:
                print(f"Converged at iteration {iteration} with residual {res_norm:.6e}")
                break

            # Update with damping for stability
            damping = 0.7
            gf_omega.vec.data += damping * a.mat.Inverse(fes_omega.FreeDofs()) * res

    print("Solving nonlinear flow system...")
    SolveNonlinearVorticity(gf_omega)

    # Step 2: Final solve for streamfunction from converged vorticity
    a_psi = BilinearForm(fes_psi)
    a_psi += grad(psi) * grad(vpsi) * dx
    a_psi.Assemble()

    l_psi = LinearForm(fes_psi)
    l_psi += gf_omega * vpsi * dx
    l_psi.Assemble()

    gf_psi = GridFunction(fes_psi)
    gf_psi.vec.data = a_psi.mat.Inverse(fes_psi.FreeDofs()) * l_psi.vec

    # Compute velocity field
    velocity = CoefficientFunction((grad(gf_psi)[1], -grad(gf_psi)[0]))

    # Visualize
    print("Visualizing solution...")
    #Draw(gf_psi, mesh, "streamfunction")
    #Draw(gf_omega, mesh, "vorticity")
    #Draw(velocity, mesh, "velocity_field", vectors={"grid_size": 30})
    speed_cf = Norm(velocity)#+grad(gf_psi)[1]  # = sqrt(u_x^2 + u_y^2)
    speed_cf = IfPos(0.5-(pi-sqrt(x**2+y**2)),pi-sqrt(x**2+y**2),0.5)*speed_cf
    return speed_cf

In [ ]:
from ngsolve import *
import numpy as np
import ngsolve
from ngsolve.webgui import Draw
ngsglobals.msg_level = 1
import ipywidgets as widgets
import matplotlib.pyplot as plt

from domain_nc import *
from functions_nc import *
#general parameters
sigma = 1 #switching parameter (0 e.g. if f is linear wrt q)
L = 1 #parameter if the functional in the u update is implemented as a simple multiplication

tstart = 0
tend = 6
tsteps = tend * int(1e3)

t = Parameter(tstart) #time parameter that will be updated
dt = (tend-tstart)/tsteps

k_u = 3 # order of Ufes, Qfes is hard coded as L2 with order 0
maxh = 0.1 # mesh size for domain

In [ ]:
# Build domain
dom = domain(maxh = maxh, order = k_u)
mesh, Ufes, Qfes = dom.mesh, dom.Ufes, dom.Qfes
print(Ufes.ndof,Qfes.ndof)
mesh.ngmesh.Save("mesh.vol")

In [ ]:
q_exact = (pi-sqrt(x**2+y**2))#cos(x)*cos(y)*cos(x-.6) #exact q
q_norm = norm(q_exact,mesh)
Draw(q_exact,mesh)

velocity = create_state(mesh)
#r = (x*(x-0.5))+(y-0.2)**2
u_exact_wot = (25*velocity)#1-(r/(pi/2)**2))**2#cos(x)*cos(y)*cos(x-0.6) #exact u without time
Draw(u_exact_wot,mesh)
u_exact = GridFunction(Ufes)#[GridFunction(Ufes) for i in range(tsteps)]
#u_exact_dt = [None for i in range (tsteps)]
#t.Set(0)
#counter = 0
#while counter<tsteps-dt:
#    decay = float((tend-t.Get())/(tend-tstart))
#    u_exact[counter].Set(u_exact_wot*decay+0.2) #exact u with time
#    t.Set(t.Get()+dt)
#    counter += 1
    #u_exact_dt = u_exact.Diff(t)  #still used???
#t.Set(0)
#Draw(u_exact[0], mesh)

z_min, z_max = 1, 4
def set_u(t,counter):
    decay = float((tend-(t.Get()+dt))/(tend-tstart))
    if counter==5000:#tsteps-1:
        #save ground truth
        np.save("u_exact.npy", u_exact.vec.FV().NumPy().copy())
        q_exact_gf = GridFunction(Qfes)
        q_exact_gf.Set(q_exact)
        np.save("q_exact.npy", q_exact_gf.vec.FV().NumPy().copy())
    return u_exact_wot*decay+z_min

#save ground truth
t.Set(0)
u_exact_gf_init = GridFunction(Ufes)
u_exact_gf_init.Set(set_u(t,counter = 0))
np.save("u_exact_init.npy", u_exact_gf_init.vec.FV().NumPy().copy())

t.Set(5)
u_exact_gf = GridFunction(Ufes)
u_exact_gf.Set(set_u(t,counter = 5000-1))
Draw(u_exact_gf, mesh, min = 1, max =3.5)
np.save("u_exact.npy", u_exact_gf.vec.FV().NumPy().copy())

q_exact_gf = GridFunction(Qfes)
q_exact_gf.Set(q_exact)
np.save("q_exact.npy", q_exact_gf.vec.FV().NumPy().copy())
#Set up Gridfunctions, that will updated trough the time steps
gfu, gfq, z, z_old, ut, u_exact_old = GridFunction(Ufes), GridFunction(Qfes), GridFunction(Ufes), GridFunction(Ufes), GridFunction(Ufes), GridFunction(Ufes)

In [ ]:
q_init = q_exact
umin, umax, qmin, qmax = 0, 3,0,3
def real_root(x, n):
    return np.sign(x) * (abs(x))**(1/n)
z_min, z_max = 0.1, 100
def C(gfq,q_exact = q_exact, q_init = q_init):
    return 2+((z_max/z_min)*norm(gfq,mesh)**(2/3)+norm(q_exact,mesh)**(2/3)+real_root(Integrate(q_init**3,mesh),3)**(2/3))**2
def absolut (gf):
    return sqrt(gf*gf)

In [ ]:
X = FESpace([Ufes,Qfes])
u, q = X.TrialFunction()
v, s = X.TestFunction()
gf = GridFunction(X)
aux = GridFunction(X) #we later write the update to this function, and then add it to gf
n = specialcf.normal(mesh.dim)
for noise_level in [0,0.05]:
    # Initial u and q
    t.Set(0)
    gfu.Set(u_exact)
    gfq.Set(0)#cos(x)*cos(y))
    gf = GridFunction(X)
    aux.vec.data[:] = 0
    #compute initial error
    error_u = []#norm(gfu-u_exact, mesh)]
    error_q = []#error_q = [norm(gfq-q_exact, mesh)]
    saved_gfu = []
    saved_gfu.append(gfu.vec.FV().NumPy().copy())
    saved_gfq = []
    saved_gfq.append(gfq.vec.FV().NumPy().copy())
    counter = 0
    gf.components[0].Set(z,definedon=mesh.Boundaries("circle"), )
    series_gfq, series_gfu = [],[]
    series_gfu.append(gfu.vec.FV().NumPy().copy())
    series_gfq.append(gfq.vec.FV().NumPy().copy())
    while t < tend - 2*dt:
        print("\r",t.Get(),"/",tend,sep="",end="")
        # Get updated data z and g
        ut = GridFunction(Ufes) #derivate in time
        u_exact_old.vec.data = u_exact.vec.Copy()
        u_exact.Set(set_u(t,counter))
        ut.vec.data = u_exact.vec.Copy()
        ut.vec.data -= u_exact_old.vec.Copy()
        ut.vec.data /= dt
        z =  add_noise(u_exact,noise_level,Ufes,mesh,20)
        zt = GridFunction(Ufes) #derivate in time
        zt.vec.data = z.vec.Copy()
        zt.vec.data -= z_old.vec.Copy()#u_exact_dt[int(t.Get()/dt)]#
        zt.vec.data /= dt
        z_old.vec.data = z.vec.Copy()
        #g = zt + f(q_exact,z) #scrap this and do it below explicitly
        #we do g in the weak form below, therefore terms including q_exact occur
        
        a = BilinearForm(X)
        res = LinearForm(X)
        
        #c eq. 
        a += q * s * dx
        a += sigma * dt * q * z * s * dx
        a += sigma * dt * (absolut(gfq)**(2/3)*q*z + q*z) * s * dx
        
        res += sigma * dt * ut * s * dx #instead of g
        res += sigma * dt * absolut(q_exact)**(2/3) * q_exact * u_exact * s * dx #instead of g
        if noise_level>0: res += sigma * dt * Grad(u_exact) * Grad(s) * dx #instead of g #cancels without noise
        if noise_level>0: res += -sigma * dt * Grad(u_exact) * n * s * ds #instead of g #cancels
        res += sigma * dt  * q_exact * z * s * dx #instead of g
        res += dt * (1+0*(5/3)*absolut(gfq)**(2/3)) * (z_old)*(gfu-z_old) * s * dx #-z
        res += -sigma * dt * (zt + absolut(gfq)**(2/3)*gfq*z + gfq*z) * s * dx
        if noise_level>0: res += -sigma * dt * Grad(z) * Grad(s) * dx #cancels without noise
        if noise_level>0: res += sigma * dt * Grad(z) * n * s * ds #cancels
        
        # u eq.
        a += u * v * dx 
        a += dt * C(gfq) * Grad(u) * Grad(v) * dx
        res += -dt * gfq * z * v * dx
        res += -dt * absolut(gfq)**(2/3)*gfq*z * v * dx

        res += -dt * absolut(gfq)**(2/3)*gfq*z * v * dx
        res += -dt * gfq * z * v * dx
        res += dt * absolut(q_exact)**(2/3) * q_exact * u_exact * v * dx#instead of g
        res += dt * zt * v * dx #instead of g
        if noise_level>0: res += dt * Grad(u_exact) * Grad(v) * dx #instead of g #cancels without noise
        if noise_level>0:res += -dt * Grad(u_exact) * n * v * ds #instead of g #cancels
        res += dt  * q_exact * u_exact * v * dx #instead of g
        if noise_level>0: res += -dt * Grad(z) * Grad(v) * dx # cancels without noise
        if noise_level>0: res += dt * Grad(z) * n * v * ds # cancels
        res += -dt * C(gfq) * Grad(gfu) * Grad(v) * dx
        res += dt * C(gfq) * Grad(z) * Grad(v) * dx
        
        a.Assemble()
        res.Assemble()  

        c = Preconditioner(a,"local")   #<-Jacobi preconditioner
        c.Update()
        solvers.BVP(bf=a, lf=res, gf=aux, pre=c, print=False)
        gf.vec.data += aux.vec.data
        
        gfu.Set(gf.components[0])
        error_q.append(norm(gfq-q_exact,mesh))
        if counter%100==0: saved_gfq.append(gfq.vec.FV().NumPy().copy())
        gfq.Set(gf.components[1])         
        error_u.append(norm(gfu-u_exact,mesh))
        if counter%100==0: saved_gfu.append(gfu.vec.FV().NumPy().copy())
        t.Set(t.Get()+ dt)    # Update time
        counter += 1
        if counter in [1,5,10,25,50,75,100,200, 500,1500,4500,5000, tsteps-2]:
            Draw(gfu, min = umin, max = umax)
            series_gfu.append(gfu.vec.FV().NumPy().copy())
            Draw(gfq, min = qmin, max = qmax)
            series_gfq.append(gfq.vec.FV().NumPy().copy())
    #print(error_u)
    #print(error_q)
    plot_error(tstart,tend,tsteps,noise_level,error_u,error_q)
    draw_gridfuncs (tsteps,u_exact,q_exact,gfu,gfq,mesh)
    save (mesh,noise_level,series_gfu,series_gfq)


In [ ]:
Draw(gfu, settings={"Objects": {"Wireframe": False}}) #
Draw(gfq, settings={"Objects": {"Wireframe": False}})

In [ ]:
Draw(u_exact[len(u_exact)-1],mesh, settings={"Objects": {"Wireframe": False}})
Draw(q_exact,mesh, settings={"Objects": {"Wireframe": False}})

In [ ]:
Draw(gfu-u_exact,mesh, settings={"Objects": {"Wireframe": False}})
Draw(gfq-q_exact,mesh, settings={"Objects": {"Wireframe": False}})

In [ ]:
#print(error_q)

In [ ]:
#print(error_u)

In [ ]:
def plot_error(tstart,tend,tsteps, noise_level,error_u,error_q):
    T_u = np.linspace(tstart,tend,len(error_u))
    T_q = np.linspace(tstart,tend,len(error_q))
    np.save("error_u "+str(noise_level)+".npy", np.array(error_u))
    np.save("error_q"+str(noise_level)+".npy", np.array(error_q))
    #general font prep
    font_path = "cmunrm.ttf"
    prop = fm.FontProperties(fname=font_path, size=15)
    prop13 = fm.FontProperties(fname=font_path, size=13)
    prop11 = fm.FontProperties(fname=font_path, size=11)

    font_path = "cmunrm.ttf"
    prop = fm.FontProperties(fname=font_path, size=15)
    prop13 = fm.FontProperties(fname=font_path, size=13)
    prop11 = fm.FontProperties(fname=font_path, size=11)

    #plot u error
    if noise_level == 0:
        plt.plot(T_u, error_u)
    else: 
        plt.plot(T_u, error_u,linewidth=0.25)
    plt.yscale('log')
    plt.xlabel("time", fontproperties = prop13)
    plt.ylabel("error", fontproperties = prop13)
    if noise_level == 0: 
        #plt.title("State error (without noise)", fontproperties = prop)
        plt.savefig("State error (without noise)_2.jpg", dpi=300)
    else: 
        #plt.title("State error (with noise)", fontproperties = prop)
        plt.savefig("State error (with noise)_2.jpg", dpi=300)
    ax = plt.gca()
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontproperties(prop11)
    plt.show()

    #plot q error
    if noise_level == 0:
        plt.plot(T_q, error_q)
    else: plt.plot(T_q,error_q,linewidth=0.25)
    plt.yscale('log')
    plt.xlabel("time", fontproperties = prop13)
    plt.ylabel("error", fontproperties = prop13)
    if noise_level == 0: 
        #plt.title("Parameter error (without noise)", fontproperties = prop)
        plt.savefig("Parameter error (without noise)_2.jpg", dpi=300)
    else: 
        #plt.title("Parameter error (with noise)", fontproperties = prop)
        plt.savefig("Parameter error (with noise)_2.jpg", dpi=300)
    ax = plt.gca()
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontproperties(prop11)
    plt.show()
plot_error(tstart,tend,tsteps, noise_level,error_u,error_q[1:])

In [ ]:
error_u_0 = np.load("error_u_0.npy")
error_q_0 = np.load("error_q_0.npy")
error_u_005 = np.load("error_u_0.05.npy")
error_q_005 = np.load("error_q_0.05.npy")
error_u_01 = np.load("error_u_0.1.npy")
error_q_01 = np.load("error_q_0.1.npy")
error_u_02 = np.load("error_u_0.2.npy")
error_q_02 = np.load("error_q_0.2.npy")

#general font prep
font_path = "cmunrm.ttf"
prop = fm.FontProperties(fname=font_path, size=15)
prop13 = fm.FontProperties(fname=font_path, size=13)
prop11 = fm.FontProperties(fname=font_path, size=11)

font_path = "cmunrm.ttf"
prop = fm.FontProperties(fname=font_path, size=15)
prop13 = fm.FontProperties(fname=font_path, size=13)
prop11 = fm.FontProperties(fname=font_path, size=11)

plt.plot(np.linspace(0,5,len(error_u_0[:5000])), error_u_0[:5000], label="State", linewidth=2.5, linestyle='dashed')
plt.plot(np.linspace(0,5,len(error_q_0[:5000])), error_q_0[:5000], label="Parameter",linewidth=2.5, linestyle='solid')
plt.yscale("log")
#plt.title("Error in u over iterations", fontproperties=prop)
plt.xlabel("time", fontproperties=prop13)
plt.ylabel("error", fontproperties=prop13)
plt.legend(prop=prop11)
plt.savefig("State and parameter 0 ac.jpg", dpi=300)
plt.show()

plt.plot(np.linspace(0,5,len(error_u_005[:5000])), error_u_005[:5000], label="State", linewidth=2.5, linestyle='dashed')
plt.plot(np.linspace(0,5,len(error_q_005[:5000])), error_q_005[:5000], label="Parameter", linewidth=2.5, linestyle='solid')
plt.yscale("log")
#plt.title("Error in u over iterations", fontproperties=prop)
plt.xlabel("time", fontproperties=prop13)
plt.ylabel("error", fontproperties=prop13)
plt.legend(prop=prop11)
plt.savefig("State and parameter 0.05 ac.jpg", dpi=300)
plt.show()

plt.plot(np.linspace(0,5,len(error_u_01[:5000])), error_u_01[:5000], label="State", linewidth=2.5, linestyle='dashed')
plt.plot(np.linspace(0,5,len(error_q_01[:5000])), error_q_01[:5000], label="Parameter", linewidth=2.5, linestyle='solid')
plt.yscale("log")
#plt.title("Error in u over iterations", fontproperties=prop)
plt.xlabel("time", fontproperties=prop13)
plt.ylabel("error", fontproperties=prop13)
plt.legend(prop=prop11)
plt.savefig("State and parameter 0.1 ac.jpg", dpi=300)
plt.show()


plt.plot(np.linspace(0,5,len(error_u_02[:5000])), error_u_02[:5000], label="State", linewidth=2.5, linestyle='dashed')
plt.plot(np.linspace(0,5,len(error_q_02[:5000])), error_q_02[:5000], label="Parameter", linewidth=2.5, linestyle='solid')
plt.yscale("log")
plt.xlabel("time", fontproperties=prop13)
plt.ylabel("error ", fontproperties=prop13)
plt.legend(prop=prop11)
plt.savefig("State and parameter 0.2 ac.jpg", dpi=300)
plt.show()  
